In [1]:
# This script makes the ternary plots and assesses fit via RMSE and R^2
using CSV
using DataFrames
using Plots
# using Statisticsrmse
include("calculate_phase_split.jl")
include("generate_training_ternary_plots.jl")

const MW = [18.01528, 96.084, 88.106]  # H2O, Furfural, EA
function mole_to_mass(x)
    w = x .* MW
    return w ./ sum(w)
end

function mass_to_mole(w)
    x = w ./ MW
    return x ./ sum(x)
end

Loading data and creating ternary diagram...
Loading data and creating ternary diagram...
\nProcessing TrainingSets/DATA_SALT_0.csv ...
Loaded 305 rows from TrainingSets/DATA_SALT_0.csv
After filtering: 305 valid rows
Successfully extracted composition data from specified columns
Calculated compositions for 305 tie lines
Converted 305 EA-rich points and 305 water-rich points to cartesian
\n=== PHASE DIAGRAM STATISTICS ===
Total tie lines: 305
Selected tie lines at indices: [1, 61, 122, 183, 244, 305]
\nEA-rich phase composition ranges:
  Water: 0.032 - 0.033
  EA: 0.784 - 0.956
  Furfural: 0.012 - 0.183
\nWater-rich phase composition ranges:
  Water: 0.907 - 0.924
  EA: 0.065 - 0.074
  Furfural: 0.002 - 0.027
\n=== DATA QUALITY CHECK ===
EA-rich phase sum check - min: 1.0, max: 1.0
Water-rich phase sum check - min: 1.0, max: 1.0
\nProcessing TrainingSets/DATA_SALT_10.csv ...
Loaded 305 rows from TrainingSets/DATA_SALT_10.csv
After filtering: 305 valid rows
Successfully extracted compos

mass_to_mole (generic function with 1 method)

In [2]:
"""
    calculate_metrics(exp_points, model_points)

Calculates RMSE and R-squared between experimental and model-predicted compositions.
Returns: (rmse_total, r2_total, rmse_by_component, r2_by_component)
"""
function calculate_metrics(exp_points, model_points)
    if length(exp_points) != length(model_points) || isempty(exp_points)
        return NaN, NaN, [NaN, NaN, NaN], [NaN, NaN, NaN]
    end
    
    # Convert to matrices for easier manipulation
    exp_matrix = hcat(exp_points...)' # Each row is a point [W, F, EA]
    model_matrix = hcat(model_points...)' # Each row is a point [W, F, EA]
    
    # Calculate overall RMSE and R²
    residuals = exp_matrix - model_matrix
    mse_total = mean(residuals.^2)
    rmse_total = sqrt(mse_total)
    
    # Overall R² (coefficient of determination)
    ss_res = sum(residuals.^2)  # Sum of squares of residuals
    ss_tot = sum((exp_matrix .- mean(exp_matrix)).^2)  # Total sum of squares
    r2_total = 1 - ss_res / ss_tot
    
    # Component-wise metrics
    rmse_by_component = [sqrt(mean((exp_matrix[:, i] - model_matrix[:, i]).^2)) for i in 1:3]
    r2_by_component = []
    
    for i in 1:3
        exp_comp = exp_matrix[:, i]
        model_comp = model_matrix[:, i]
        ss_res_comp = sum((exp_comp - model_comp).^2)
        ss_tot_comp = sum((exp_comp .- mean(exp_comp)).^2)
        r2_comp = 1 - ss_res_comp / ss_tot_comp
        push!(r2_by_component, r2_comp)
    end
    
    return rmse_total, r2_total, rmse_by_component, r2_by_component
end

calculate_metrics

In [3]:
"""
    print_metrics_summary(salt_conc, rmse_total, r2_total, rmse_by_component, r2_by_component)

Prints formatted metrics summary for a given salt concentration.
"""
function print_metrics_summary(salt_conc, rmse_total, r2_total, rmse_by_component, r2_by_component)
    component_names = ["Water", "Furfural", "Ethyl Acetate"]
    
    println("Metrics for $salt_conc:")
    println("  Overall RMSE: $(round(rmse_total, digits=4))")
    println("  Overall R²:   $(round(r2_total, digits=4))")
    println("  Component-wise RMSE:")
    for (i, comp) in enumerate(component_names)
        println("    $comp: $(round(rmse_by_component[i], digits=4))")
    end
    println("  Component-wise R²:")
    for (i, comp) in enumerate(component_names)
        println("    $comp: $(round(r2_by_component[i], digits=4))")
    end
    println()
end

print_metrics_summary

In [3]:
"""
    create_validation_plot(exp_tie_lines, model_tie_lines, salt_conc, rmse_total, r2_total)

Creates a ternary plot comparing experimental and model tie lines with metrics.
"""
function create_validation_plot(exp_tie_lines, model_tie_lines, salt_conc, rmse_total, r2_total)
    plt = plot(size=(800, 750), dpi=300)
    draw_ternary_triangle_clean!()
    add_ternary_grid!(0.1)

    # Separate tie line endpoints by phase type (water-rich vs EA-rich)
    exp_water_rich_cart = []
    exp_ea_rich_cart = []
    model_water_rich_cart = []
    model_ea_rich_cart = []

    # Plot Experimental Tie Lines and identify phases
    for (i_plot, (p1, p2)) in enumerate(exp_tie_lines)
        cart_p1 = ternary_to_cartesian(p1[1], p1[3], p1[2]) # W, EA, F
        cart_p2 = ternary_to_cartesian(p2[1], p2[3], p2[2]) # W, EA, F
        
        # Identify which phase is water-rich vs EA-rich
        if p1[1] > p2[1]  # p1 has more water, so p1 is water-rich
            push!(exp_water_rich_cart, cart_p1)
            push!(exp_ea_rich_cart, cart_p2)
        else  # p2 has more water, so p2 is water-rich
            push!(exp_water_rich_cart, cart_p2)
            push!(exp_ea_rich_cart, cart_p1)
        end
        
        # Plot tie line
        plot!([cart_p1[1], cart_p2[1]], [cart_p1[2], cart_p2[2]], 
              color=:blue, linewidth=2, label=(i_plot==1 ? "Reference" : ""))
    end

    # Plot Model Tie Lines and identify phases
    for (i_plot, (p1, p2)) in enumerate(model_tie_lines)
        cart_p1 = ternary_to_cartesian(p1[1], p1[3], p1[2]) # W, EA, F
        cart_p2 = ternary_to_cartesian(p2[1], p2[3], p2[2]) # W, EA, F
        
        # Identify which phase is water-rich vs EA-rich
        if p1[1] > p2[1]  # p1 has more water, so p1 is water-rich
            push!(model_water_rich_cart, cart_p1)
            push!(model_ea_rich_cart, cart_p2)
        else  # p2 has more water, so p2 is water-rich
            push!(model_water_rich_cart, cart_p2)
            push!(model_ea_rich_cart, cart_p1)
        end
        
        # Plot tie line
        plot!([cart_p1[1], cart_p2[1]], [cart_p1[2], cart_p2[2]], 
              color=:red, linestyle=:dash, linewidth=2, 
              label=(i_plot==1 ? "Predicted" : ""))
    end

    # Add markers for experimental tie line endpoints
    if !isempty(exp_water_rich_cart)
        # Water-rich phase - black squares
        scatter!([p[1] for p in exp_water_rich_cart], [p[2] for p in exp_water_rich_cart],
                markershape=:square, markersize=8, markercolor=:black,
                markerstrokecolor=:blue, markerstrokewidth=2,
                label="Exp. Water-rich")
        
        # EA-rich phase - white squares  
        scatter!([p[1] for p in exp_ea_rich_cart], [p[2] for p in exp_ea_rich_cart],
                markershape=:square, markersize=8, markercolor=:white,
                markerstrokecolor=:blue, markerstrokewidth=2,
                label="Exp. EA-rich")
    end

    # Add markers for model tie line endpoints
    if !isempty(model_water_rich_cart)
        # Water-rich phase - black circles
        scatter!([p[1] for p in model_water_rich_cart], [p[2] for p in model_water_rich_cart],
                markershape=:circle, markersize=8, markercolor=:black,
                markerstrokecolor=:red, markerstrokewidth=2,
                label="Model Water-rich")
        
        # EA-rich phase - white circles
        scatter!([p[1] for p in model_ea_rich_cart], [p[2] for p in model_ea_rich_cart],
                markershape=:circle, markersize=8, markercolor=:white,
                markerstrokecolor=:red, markerstrokewidth=2,
                label="Model EA-rich")
    end
    
    # Add metrics to plot title
    metrics_text = "RMSE: $(round(rmse_total, digits=4)), R²: $(round(r2_total, digits=4))"
    plot!(title="Validation for $salt_conc Data\n$metrics_text", titlefontsize=12,
          legend=:topright, legendfontsize=8)
    
    return plt
end

create_validation_plot

In [4]:
# Define file paths and salt concentrations
tau_files = [
    "fitted_tau_matrix_0.csv", "fitted_tau_matrix_1_75.csv",
    "fitted_tau_matrix_2_5.csv", "fitted_tau_matrix_3_75.csv",
    "fitted_tau_matrix_5.csv", "fitted_tau_matrix_7_5.csv",
    "fitted_tau_matrix_10.csv"
]

data_files = [
    "DATA_SALT_0.csv", "DATA_SALT_1_75.csv",
    "DATA_SALT_2_5.csv", "DATA_SALT_3_75.csv",
    "DATA_SALT_5.csv", "DATA_SALT_7_5.csv",
    "DATA_SALT_10.csv"
]

salt_concentrations = ["0% NaCl", "1.75% NaCl", "2.5% NaCl", "3.75% NaCl", 
                      "5% NaCl", "7.5% NaCl", "10% NaCl"]

println("Configuration loaded for $(length(tau_files)) salt concentrations")

Configuration loaded for 7 salt concentrations


In [9]:
println("Starting batch validation analysis...")
println("-"^70)

# Storage for all metrics
all_metrics = []

for (i, tau_file) in enumerate(tau_files)
    data_file = data_files[i]
    salt_conc = salt_concentrations[i]
    
    println("Processing: $salt_conc")

    # Load data and model
    τ_matrix = read_tau_matrix(tau_file)
    aspen_df = CSV.read(joinpath("TrainingSets", data_file), DataFrame)

    # Prepare data
    total_flows = aspen_df.FLOWWA .+ aspen_df.FLOWEA .+ aspen_df.FLOWFUR
    z_feeds_mass = [[w, f, e] for (w, f, e) in zip(aspen_df.FLOWWA ./ total_flows, 
                                             aspen_df.FLOWFUR ./ total_flows, 
                                             aspen_df.FLOWEA ./ total_flows)]
    z_feeds = [mass_to_mole(z) for z in z_feeds_mass]  # Convert to mole fractions
    
    num_tie_lines = length(z_feeds)
    indices_to_plot = round.(Int, range(1, num_tie_lines - 1, length=6)) |> unique

    # Calculate predictions and collect data
    model_points_for_metrics = []
    exp_points_for_metrics = []
    model_tie_lines_for_plot = []
    exp_tie_lines_for_plot = []

    exp_p1_water = aspen_df.XWIN1; exp_p1_fur = aspen_df.XFIN1; exp_p1_ea = aspen_df.XEAIN1
    exp_p2_water = aspen_df.XWIN2; exp_p2_fur = aspen_df.XFIN2; exp_p2_ea = aspen_df.XEAIN2

    for idx in indices_to_plot
        # Experimental data
        exp_p1_mass = [(exp_p1_water[idx]+exp_p1_water[idx+1])/2, (exp_p1_fur[idx]+exp_p1_fur[idx+1])/2, (exp_p1_ea[idx]+exp_p1_ea[idx+1])/2]
        exp_p2_mass = [(exp_p2_water[idx]+exp_p2_water[idx+1])/2, (exp_p2_fur[idx]+exp_p2_fur[idx+1])/2, (exp_p2_ea[idx]+exp_p2_ea[idx+1])/2]
        exp_p1 = mass_to_mole(exp_p1_mass)
        exp_p2 = mass_to_mole(exp_p2_mass)
        push!(exp_tie_lines_for_plot, (exp_p1, exp_p2))

        z1 = z_feeds[idx]
        z2 = z_feeds[idx + 1]
        z_intermediate = (z1 .+ z2) ./ 2

        # Model predictions for metrics
        model_p1_metric, model_p2_metric = solve_lle(z_intermediate, τ_matrix)
        if !any(isnan, model_p1_metric)
            # Phase matching for metrics
            if exp_p1[1] > exp_p2[1]
                exp_water = exp_p1
                exp_ea = exp_p2
            else
                exp_water = exp_p2
                exp_ea = exp_p1
            end
            if model_p1_metric[1] > model_p2_metric[1]
                model_water = model_p1_metric
                model_ea = model_p2_metric
            else
                model_water = model_p2_metric
                model_ea = model_p1_metric
            end
            push!(exp_points_for_metrics, exp_water, exp_ea)
            push!(model_points_for_metrics, model_water, model_ea)
        end

        
        model_p1_plot, model_p2_plot = solve_lle(z_intermediate, τ_matrix)
        if !any(isnan, model_p1_plot)
            push!(model_tie_lines_for_plot, (model_p1_plot, model_p2_plot))
        end
    end

    # Calculate metrics
    rmse_total, r2_total, rmse_by_component, r2_by_component = 
        calculate_metrics(exp_points_for_metrics, model_points_for_metrics)
    
    # Store metrics
    push!(all_metrics, (salt_conc, rmse_total, r2_total, rmse_by_component, r2_by_component))
    
    # Create and save plot
    plt = create_validation_plot(exp_tie_lines_for_plot, model_tie_lines_for_plot, 
                               salt_conc, rmse_total, r2_total)
    
    plot_name = "validation_plot_test$(replace(data_file, "DATA_SALT_interpolatedfeeds_" => "", ".csv" => "_NaCl", "." => "_")).png"
    savefig(plt, joinpath(output_dir, plot_name))
    
    println("  RMSE: $(round(rmse_total, digits=4)), R²: $(round(r2_total, digits=4))")
    println("  Plot saved: $plot_name")
    println()
end

println("Batch analysis completed!")

Starting batch validation analysis...
----------------------------------------------------------------------
Processing: 0% NaCl
Loaded tau matrix from fitted_tau_matrix_0.csv:


3×3 Matrix{Float64}:
 0.0      4.46684   4.7406
 2.83952  0.0       3.9569
 2.75124  0.809533  0.0

┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441
┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441
┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5, 6]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441
┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5, 6]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441
┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ E

  RMSE: 0.0029, R²: 0.9999
  Plot saved: validation_plot_testDATA_SALT_0_NaCl.png

Processing: 1.75% NaCl
Loaded tau matrix from fitted_tau_matrix_1_75.csv:


3×3 Matrix{Float64}:
 0.0      4.12335    4.30973
 2.0518   0.0       -0.552372
 2.85893  0.898107   0.0

...

3×3 Matrix{Float64}:
 0.0      4.02966  3.92115
 2.47271  0.0      2.12934
 3.33982  1.97005  0.0

┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441
┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441


┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5, 6]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441


┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5, 6]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441
┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5, 6]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441
┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5, 6]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441
┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.
│ Element indices affected: [5]
└ @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/fminbox.jl:441
┌ Warning: Initial position cannot be on the boundary of the box. Moving elements to the interior.


  RMSE: 0.0852, R²: 0.9516
  Plot saved: validation_plot_testDATA_SALT_10_NaCl.png

Batch analysis completed!


In [10]:
# Display comprehensive results summary
println("-"^80)
println("COMPREHENSIVE VALIDATION RESULTS SUMMARY")
println("-"^80)
println("Salt Conc.     Overall RMSE    Overall R²     Water RMSE    Furfural RMSE    EA RMSE")
println("-"^80)

for (salt_conc, rmse_total, r2_total, rmse_by_component, r2_by_component) in all_metrics
    formatted_salt = rpad(salt_conc, 12)
    formatted_rmse = rpad(string(round(rmse_total, digits=4)), 13)
    formatted_r2 = rpad(string(round(r2_total, digits=4)), 13)
    formatted_w_rmse = rpad(string(round(rmse_by_component[1], digits=4)), 12)
    formatted_f_rmse = rpad(string(round(rmse_by_component[2], digits=4)), 15)
    formatted_ea_rmse = string(round(rmse_by_component[3], digits=4))
    
    println("$formatted_salt $formatted_rmse $formatted_r2 $formatted_w_rmse $formatted_f_rmse $formatted_ea_rmse")
end

println("-"^80)

# Calculate and display average metrics
valid_metrics = [(m[2], m[3]) for m in all_metrics if !isnan(m[2]) && !isnan(m[3])]
avg_rmse = mean([m[1] for m in valid_metrics])
avg_r2 = mean([m[2] for m in valid_metrics])

println("OVERALL PERFORMANCE:")
println("Average RMSE across all salt concentrations: $(round(avg_rmse, digits=4))")
println("Average R² across all salt concentrations:   $(round(avg_r2, digits=4))")

# Identify best and worst performing conditions
rmse_values = [m[2] for m in all_metrics if !isnan(m[2])]
r2_values = [m[3] for m in all_metrics if !isnan(m[3])]

if !isempty(rmse_values)
    best_rmse_idx = argmin(rmse_values)
    worst_rmse_idx = argmax(rmse_values)
    best_r2_idx = argmax(r2_values)
    worst_r2_idx = argmin(r2_values)
    
    println("\nPERFORMANCE HIGHLIGHTS:")
    println("Best RMSE:  $(all_metrics[best_rmse_idx][1]) (RMSE = $(round(rmse_values[best_rmse_idx], digits=4)))")
    println("Worst RMSE: $(all_metrics[worst_rmse_idx][1]) (RMSE = $(round(rmse_values[worst_rmse_idx], digits=4)))")
    println("Best R²:    $(all_metrics[best_r2_idx][1]) (R² = $(round(r2_values[best_r2_idx], digits=4)))")
    println("Worst R²:   $(all_metrics[worst_r2_idx][1]) (R² = $(round(r2_values[worst_r2_idx], digits=4)))")
end

println("-"^80)

--------------------------------------------------------------------------------
COMPREHENSIVE VALIDATION RESULTS SUMMARY
--------------------------------------------------------------------------------
Salt Conc.     Overall RMSE    Overall R²     Water RMSE    Furfural RMSE    EA RMSE
--------------------------------------------------------------------------------
0% NaCl      0.0029        0.9999        0.0028       0.0017          0.0039
1.75% NaCl   0.0809        0.9585        0.1          0.0046          0.098
2.5% NaCl    0.1103        0.9227        0.1392       0.0159          0.1299
3.75% NaCl   0.1098        0.9227        0.1386       0.0142          0.1295
5% NaCl      0.0088        0.9995        0.0095       0.0056          0.0105
7.5% NaCl    0.011         0.9992        0.0127       0.0047          0.0134
10% NaCl     0.0852        0.9516        0.1044       0.0042          0.1043
--------------------------------------------------------------------------------
OVERALL PERF